In [ ]:
import Pkg; Pkg.add("QuantumCumulants")


In [ ]:
import Pkg; Pkg.add("ModelingToolkit")


In [ ]:
import Pkg; Pkg.add("OrdinaryDiffEq")

In [ ]:
using QuantumCumulants
using OrdinaryDiffEq
using ModelingToolkit

# Define Lattice Parameters
Lx, Ly = 2, 2
N = Lx * Ly
J = 0.1         # Hopping
g = 0.05        # Coupling
kappa = 1.0     # Decay rate

# Define the Unit Cell 
# We create ONE Hilbert space definition for a generic site
h_cav = ⊗([FockSpace(Symbol(:cavity, i)) for i = 1:N]...)
h_atom = ⊗([NLevelSpace(Symbol(:atom, i), 2) for i = 1:N]...)

h_site = h_cav ⊗ h_atom # The space of one site

# Define operators for this generic site
# a_unit: acts on component 1 (cavity)
a_unit = Destroy(h_site, :a, 1) 
# σ_unit: acts on component 2 (atom), transition g(1)->e(2)
σ_unit = Transition(h_site, :σ, 2, 1, 2) 

# Create Lattice Operators (Indexed)
# This creates a wrapper so a(i) is the operator on site i
a(i) = IndexedOperator(a_unit, i)
σ(i) = IndexedOperator(σ_unit, i)
σz(i) = IndexedOperator(Transition(h_site, :σz, 2, 2, 2) - Transition(h_site, :σz, 1, 1, 2), i)

# Construct Hamiltonian 
# We can just sum over indices, and the library knows a(i) and a(j) are compatible
H = 0
for i in 1:N
    # Local Jaynes-Cummings
    # Note: We must use the indexed operators a(i) and σ(i)
    H += 1.0 * a(i)' * a(i) + 1.0 * σz(i) + g * (a(i)' * σ(i) + a(i) * σ(i)')
    
    # Hopping (Nearest Neighbors)
    x = (i-1) % Lx
    y = div(i-1, Lx)
    
    # Hop Right
    if x < Lx-1
        j = i + 1
        H += -J * (a(i)' * a(j) + a(i) * a(j)')
    end
    # Hop Up
    if y < Ly-1
        j = i + Lx
        H += -J * (a(i)' * a(j) + a(i) * a(j)')
    end
end

# Derive Mean Field Equations
# We want to track <a_i> and <σ_i> for all sites
ops_list = [a(i) for i in 1:N]
append!(ops_list, [σ(i) for i in 1:N])

println("Deriving equations... (this may take a moment)")
eqs = meanfield(ops_list, H; order=1) 

# Solve 
# Initial state: Put 1 photon in site 1 (real part=1.0)
u0 = zeros(ComplexF64, length(eqs))
u0[1] = 1.0 + 0.0im # Set <a_1> = 1

@named sys = ODESystem(eqs)
prob = ODEProblem(sys, u0, (0.0, 10.0))
sol = solve(prob, Tsit5())

# Plotting 
using Plots
# Plot photon number expectation |<a>|^2 approx for first few sites
plot(sol.t, abs2.(sol[a(1)]), label="Site 1 (Source)")
plot!(sol.t, abs2.(sol[a(2)]), label="Site 2")

In [ ]:
using QuantumCumulants
using OrdinaryDiffEq
using ModelingToolkit

In [ ]:
Lx, Ly = 2, 2
N = Lx * Ly
J = 0.1         # Hopping
g = 0.05        # Coupling
kappa = 1.0     # Decay rate



In [ ]:
h_cav = ⊗([FockSpace(Symbol(:cavity, i)) for i = 1:N]...)
h_atom = ⊗([NLevelSpace(Symbol(:atom, i), 2) for i = 1:N]...)

h_site = h_cav ⊗ h_atom # The space of one site

In [ ]:
using QuantumCumulants
using OrdinaryDiffEq
using ModelingToolkit

# --- 1. Define Lattice Parameters 
Lx, Ly = 2, 2
N = Lx * Ly
J = 0.1         # Hopping
g = 0.05        # Coupling

# --- 2. Supervisor's Giant Space Definition ---
# Creates a tensor product of N cavities, then N atoms. Total = 2N components.
h_cav = ⊗([FockSpace(Symbol(:cavity, i)) for i = 1:N]...)
h_atom = ⊗([NLevelSpace(Symbol(:atom, i), 2) for i = 1:N]...)

h_total = h_cav ⊗ h_atom 

# --- 3. Create Explicit Operator Arrays  ---
# Cavities are in components 1 through N
a = [Destroy(h_total, Symbol(:a, i), i) for i in 1:N]

# Atoms are in components (N+1) through 2N
σ  = [Transition(h_total, Symbol(:σ, i), 2, 1, N + i) for i in 1:N]
σz = [Transition(h_total, Symbol(:σz, i), 2, 2, N + i) - 
      Transition(h_total, Symbol(:σz, i), 1, 1, N + i) for i in 1:N]

# --- 4. Construct Hamiltonian ---
H = 0
for i in 1:N
    # Local Jaynes-Cummings
    H += 1.0 * a[i]' * a[i] + 1.0 * σz[i] + g * (a[i]' * σ[i] + a[i] * σ[i]')
    
    # Hopping (Nearest Neighbors)
    x = (i-1) % Lx
    y = div(i-1, Lx)
    
    # Hop Right
    if x < Lx-1
        j = i + 1
        H += -J * (a[i]' * a[j] + a[i] * a[j]')
    end
    # Hop Up
    if y < Ly-1
        j = i + Lx
        H += -J * (a[i]' * a[j] + a[i] * a[j]')
    end
end

# 5. Derive Mean Field Equations ---
ops_list = [a; σ; σz]
println("Deriving equations")
eqs = meanfield(ops_list, H; order=1) 

# 6. Solve
u0 = zeros(ComplexF64, length(eqs))
u0[1] = 1.0 + 0.0im # Set <a_1> = 1 
u0[end-N+1:end] .= -1.0 + 0.0im # Set all <σz> = 0 (equal superposition)

@named sys = ODESystem(eqs)
prob = ODEProblem(sys, u0, (0.0, 10.0))
sol = solve(prob, Tsit5())

# 7. Plotting
using Plots
plot(sol.t, abs2.(sol[a[1]]), label="Site 1 (Source)")
plot!(sol.t, abs2.(sol[a[2]]), label="Site 2")

In [ ]:
using CUDA
using QuantumToolbox

# 1. Verify the GPU is visible to Julia
println("CUDA Functional: ", CUDA.functional())
println("CUDA Version: ", CUDA.version())

# 2. Create a simple state on the CPU (e.g., a 5-level system with 2 photons)
cpu_state = fock(5, 2) 

# 3. Transfer the state to the GPU
gpu_state = cu(cpu_state)

println("\nSuccess! State transferred to: ", typeof(gpu_state))